# Training

In [1]:
!pip install -q ml-collections

In [2]:
import tensorflow as tf
# Set the device to CPU
tf.config.set_visible_devices([], 'GPU')

2026-04-30 20:39:08.569359: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777581548.775756      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777581548.835089      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777581549.309795      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777581549.309838      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777581549.309841      23 computation_placer.cc:177] computation placer alr

In [3]:
import os
import urllib.request
from urllib.error import HTTPError
import ml_collections
import jax
from jax import numpy as jnp

main_rng_key = jax.random.key(18)

In [4]:
!rm -rf tokenizer_32_000_vocab_size_model
!rm -rf log_dir
!rm -f configs.py tokenizer.py data.py model.py training_utils.py


!mkdir tokenizer_32_000_vocab_size_model

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [5]:
base_url = "https://raw.githubusercontent.com/MiguelSteph/transformer-from-scratch/version2/"
def download_file_from_github(file_path: str, file_name: str):
    if not os.path.isfile(file_name):
        file_url = base_url + file_path
        print(f"Downloading {file_url}...")
        try:
            urllib.request.urlretrieve(file_url, file_name)
        except HTTPError as e:
            print("Something went wrong. Please try to download the file directly from the GitHub repository:\n", e)

file_paths = [
    'configs/configs.py',
    'data/tokenizer.py',
    'data/data.py',
    'models/model.py',
    'training/training_utils.py',
    'data/tokenizer_32_000_vocab_size_model/merges.txt',
    'data/tokenizer_32_000_vocab_size_model/vocab.json',
]
file_names = [
    'configs.py',
    'tokenizer.py',
    'data.py',
    'model.py',
    'training_utils.py',
    'tokenizer_32_000_vocab_size_model/merges.txt',
    'tokenizer_32_000_vocab_size_model/vocab.json',
]

for file_path, file_name in zip(file_paths, file_names): 
    download_file_from_github(file_path, file_name)

In [6]:
from configs import get_configs
from model import create_transformer_module
from training_utils import train_and_evaluate, get_dataset_iterator, create_train_state, generate_random_batch, train_step
from data import load_preprocessed_dataset

base_configs = get_configs()
config = ml_collections.ConfigDict(base_configs)
config.data.test_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/test.tfrecord'
config.data.validation_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/validation.tfrecord'
config.data.train_ds_path = '/kaggle/input/datasets/migsena/de-en-preprocessed-dataset/train.tfrecord'

train_ds = load_preprocessed_dataset(config.data.train_ds_path, config.data.max_seq_len)
validation_ds = load_preprocessed_dataset(config.data.validation_ds_path, config.data.max_seq_len)
test_ds = load_preprocessed_dataset(config.data.test_ds_path, config.data.max_seq_len)

In [7]:
!rm -rf log_dir 
!rm -f log_dir.zip

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [8]:
config

data:
  batch_size: 32
  de_tokenizer_model_path: tokenizer_32_000_vocab_size_model
  en_tokenizer_model_path: tokenizer_32_000_vocab_size_model
  max_seq_len: 100
  special_tokens:
  - <|startoftext|>
  - <|endoftext|>
  test_ds_path: /kaggle/input/datasets/migsena/de-en-preprocessed-dataset/test.tfrecord
  tokenizer_model_path: tokenizer_32_000_vocab_size_model
  train_ds_path: /kaggle/input/datasets/migsena/de-en-preprocessed-dataset/train.tfrecord
  validation_ds_path: /kaggle/input/datasets/migsena/de-en-preprocessed-dataset/validation.tfrecord
  vocab_size: 32000
model:
  d_proj: 128
  dropout: 0.1
  emb_dim: 128
  ff_d_inner_factor: 2
  num_blocks: 4
  num_heads: 6
optimizer:
  base_lr: 0.0005
  steps_per_epochs: 15000
  training_epochs: 30
  warmup_epochs: 4
training_output:
  checkpoint_path: log_dir/checkpoints
  metric_path: log_dir/metrics
  trace_path: log_dir/traces

# Training

In [9]:
model = create_transformer_module(config)
state = train_and_evaluate(model, 
                           config,
                           main_rng_key,
                           train_ds,
                           validation_ds,
                           log_dir_prefix=None)

/kaggle/working/training_utils.py:50: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'>  is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  enc_input = jax.random.randint(key=prng_1, shape=(batch_size, max_seq_len),
/kaggle/working/training_utils.py:52: UserWarning: Explicitly requested dtype <class 'jax.numpy.int64'>  is not available, and will be truncated to dtype int32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/jax-ml/jax#current-gotchas for more.
  dec_input_raw = jax.random.randint(key=prng_2, shape=(batch_size, max_seq_len+1),


Epoch 1


  0%|          | 0/15000 [00:00<?, ?it/s]

Training:    Loss: 9.80716609954834    Accuracy: 0.13511024415493011
Validation:  Loss: 8.567076683044434    Accuracy: 0.156980499625206
Epoch 2


  0%|          | 0/15000 [00:00<?, ?it/s]

...

Training:    Loss: 2.859323024749756    Accuracy: 0.5052554607391357
Validation:  Loss: 2.8648946285247803    Accuracy: 0.5095649361610413
Epoch 29


  0%|          | 0/15000 [00:00<?, ?it/s]

Training:    Loss: 2.8410897254943848    Accuracy: 0.5071418285369873
Validation:  Loss: 2.8927927017211914    Accuracy: 0.5072140693664551
Epoch 30


  0%|          | 0/15000 [00:00<?, ?it/s]

Training:    Loss: 2.8317365646362305    Accuracy: 0.5079945921897888
Validation:  Loss: 2.826817035675049    Accuracy: 0.5128915309906006
